## Notebook 04: Tuning Qwen further

With 03 scoring 0.973, we can try pseudo-labeling the test set, then running a bigger version Qwen

**Model:** [`Qwen/Qwen2.5-0.5B`](https://huggingface.co/Qwen/Qwen2.5-0.5B) (Apache-2.0)

## Setup
We can again run this on a Kaggle T4 GPU

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "peft"], check=True)

if importlib.util.find_spec("torchao") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

In [2]:
import glob
import os
import time
from pathlib import Path

import certifi
import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import LoraConfig, TaskType, get_peft_model
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SEED = 77

ON_KAGGLE = Path("/kaggle/input").exists()
DATA_DIR = Path(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]).parent if ON_KAGGLE else Path("../data")
RESULTS_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../results/04")
SUBMISSION_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../submissions")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

pd.set_option("display.precision", 3)
transformers.logging.set_verbosity_error()
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}")
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE)

W1108 19:15:11.360000 68028 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


torch 2.15.0, transformers 5.22.1, peft 0.21.2
Device: mps


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
y = train["label"].values

## 1. Teacher + pseudo-labels

We can use  03 as a teacher and take test passages that it's confident about (P(AI) ≥ 0.95 or ≤ 0.05) as a pseudo-label, and retrain on the training set + those passages

So it can't just repeat a pseudo-label it trained on, we can split test into two halves: one model learns from half A's pseudo-labels and predicts half B, the other does the reverse

In [4]:
TEACHER_FILE = "results/03/03_qwen2.5-0.5b_test_by_fold.csv"
TEACHER_COMMIT = "a71bc0e6d425eec3bd182bf36bb0f0f2964a90fa"

teacher_src = Path("..") / TEACHER_FILE
if not teacher_src.exists():
    teacher_src = f"https://raw.githubusercontent.com/rkoren/ai-vs-human-text-kaggle/{TEACHER_COMMIT}/{TEACHER_FILE}"

teacher = test[["id"]].merge(pd.read_csv(teacher_src), on="id")
assert len(teacher) == len(test)
teacher_p = teacher.filter(like="fold_").mean(axis=1).values

In [5]:
PL_THRESHOLD = 0.95 # can adjust
confident = (teacher_p >= PL_THRESHOLD) | (teacher_p <= 1 - PL_THRESHOLD)
pseudo_label = (teacher_p >= 0.5).astype(int)

half_a, half_b = train_test_split(np.arange(len(test)), test_size=0.5, random_state=SEED,
                                  stratify=pd.qcut(teacher_p, 10, labels=False, duplicates="drop"))

pd.DataFrame({
    "pseudo-labeled AI": [(confident & (pseudo_label == 1))[h].sum() for h in (half_a, half_b)],
    "pseudo-labeled human": [(confident & (pseudo_label == 0))[h].sum() for h in (half_a, half_b)],
    "uncertain (left out)": [(~confident)[h].sum() for h in (half_a, half_b)],
}, index=["half A", "half B"])

,pseudo-labeled AI,pseudo-labeled human,uncertain (left out)
half A,653,666,181
half B,647,668,185


## 2. Qwen

Same model and settings as 03 (LoRA r=16, lr 1e-4, 1 epoch, batch 8, 512 tokens), but the only change is that training takes token ids and labels directly, so pseudo-labeled test passages can be added

In [6]:
MAX_LENGTH = 512
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 1
USE_FP16 = DEVICE == "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def make_qwen(model_name=MODEL_NAME, checkpointing=False):
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
    model.config.pad_token_id = tokenizer.pad_token_id
    model = model.float()
    if checkpointing:
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.0,
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora).to(DEVICE)
    assert all(p.dtype == torch.float32 for p in model.parameters())
    return model

train_ids = tokenizer(list(train["text"]), truncation=True, max_length=MAX_LENGTH)["input_ids"]
test_ids = tokenizer(list(test["text"]), truncation=True, max_length=MAX_LENGTH)["input_ids"]
print(f"Tokens per passage: median {np.median([len(t) for t in train_ids]):.0f}, "
      f"truncated at {MAX_LENGTH}: {sum(len(t) >= MAX_LENGTH for t in train_ids)}")

Tokens per passage: median 278, truncated at 512: 72


In [7]:
def batches(ids, idx, rng=None):
    """Batches of similar-length passages (less padding). Batch order is shuffled when an rng is given."""
    idx = np.asarray(idx)
    order = idx[np.argsort([len(ids[i]) for i in idx], kind="stable")]
    chunks = [order[start:start + BATCH_SIZE] for start in range(0, len(order), BATCH_SIZE)]
    if rng is not None:
        rng.shuffle(chunks)
    for chunk in chunks:
        enc = tokenizer.pad({"input_ids": [ids[i] for i in chunk]}, return_tensors="pt")
        yield chunk, {k: v.to(DEVICE) for k, v in enc.items()}

def train_qwen(ids, labels, seed, model_name=MODEL_NAME, checkpointing=False):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_qwen(model_name, checkpointing)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE)
    n_steps = EPOCHS * int(np.ceil(len(ids) / BATCH_SIZE))
    warmup = max(1, int(0.1 * n_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min((s + 1) / warmup, (n_steps - s) / (n_steps - warmup)))
    scaler = torch.amp.GradScaler(enabled=USE_FP16)
    labels = torch.tensor(labels, dtype=torch.float32)

    model.train()
    step, recent_losses = 0, []
    for epoch in range(EPOCHS):
        for chunk, enc in batches(ids, np.arange(len(ids)), rng):
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
                logits = model(**enc).logits.squeeze(-1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), labels[chunk].to(DEVICE))
            assert torch.isfinite(loss), f"loss is {loss.item()} at step {step}"
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            recent_losses.append(loss.item())
            if step % 50 == 0 or step == n_steps:
                print(f"  step {step}/{n_steps}  avg loss (last 50) {np.mean(recent_losses[-50:]):.3f}")
    return model

@torch.no_grad()
def predict_qwen(model, ids):
    model.eval()
    preds = np.full(len(ids), np.nan)
    for chunk, enc in batches(ids, np.arange(len(ids))):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
            preds[chunk] = torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
    return preds

def fit_predict(ids, labels, predict_ids, **kwargs):
    model = train_qwen(ids, labels, seed=SEED, **kwargs)
    preds = predict_qwen(model, predict_ids)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return preds

Test predictions are saved to `results/04/` and committed, so the notebook loads them instead of retraining. We should delete them after changing any settings, otherwise the old predictions get loaded

In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def cached_run(name, fit):
    """Loads this run's saved test predictions, or trains and saves them"""
    path = RESULTS_DIR / f"04_{name}_test.csv"
    if path.exists():
        print(f"{name}: loading saved predictions from {RESULTS_DIR}")
        return test[["id"]].merge(pd.read_csv(path), on="id")["label"].values
    start = time.time()
    preds = fit()
    assert len(preds) == len(test) and not np.isnan(preds).any()
    pd.DataFrame({"id": test["id"], "label": preds}).to_csv(path, index=False)
    print(f"{name}: trained in {(time.time() - start) / 60:.0f} min")
    return preds

## 3. Runs

**All train:** same as 03, but one model on all 10,000 train passages (the control)

**Pseudo-label:** the same, plus the pseudo-labeled half of test, predicting the other half

In [9]:
control = cached_run("qwen2.5-0.5b_all-train", lambda: fit_predict(train_ids, y, test_ids))

qwen2.5-0.5b_all-train: loading saved predictions from ../results/04


In [10]:
def cross_fit():
    preds = np.full(len(test), np.nan)
    for learn_from, predict in [(half_b, half_a), (half_a, half_b)]:
        pl_rows = learn_from[confident[learn_from]]
        assert not set(pl_rows) & set(predict)  # never predicts a passage it saw a pseudo-label on
        print(f"Training on {len(train_ids)} train + {len(pl_rows)} pseudo-labeled, predicting {len(predict)} test passages")
        ids = train_ids + [test_ids[i] for i in pl_rows]
        labels = np.concatenate([y, pseudo_label[pl_rows]])
        preds[predict] = fit_predict(ids, labels, [test_ids[i] for i in predict])
    return preds

pseudo = cached_run("qwen2.5-0.5b_pseudo-label", cross_fit)

qwen2.5-0.5b_pseudo-label: loading saved predictions from ../results/04


## 4. Compare

There are no test labels, so here we can only see how the predictions moved

In [11]:
preds = {"03 (5-fold average)": teacher_p, "all train": control, "pseudo-label": pseudo}
spearman = lambda a, b: pd.Series(a).corr(pd.Series(b), method="spearman")

pd.DataFrame({
    "corr. with 03": {name: spearman(p, teacher_p) for name, p in preds.items()},
    "test % predicted AI": {name: (p > 0.5).mean() * 100 for name, p in preds.items()},
    "mean change on uncertain passages": {name: np.abs(p - teacher_p)[~confident].mean() for name, p in preds.items()},
})

,corr. with 03,test % predicted AI,mean change on uncertain passages
03 (5-fold average),1.000,47.267,0.000
all train,0.944,49.567,0.282
pseudo-label,0.935,47.300,0.277


## 5. Submissions

In [12]:
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
SUBMISSION_DIR.mkdir(exist_ok=True)
for name, filename in [("all train", "04_qwen_all_train.csv"), ("pseudo-label", "04_qwen_pseudo_label.csv")]:
    submission = pd.DataFrame({"id": test["id"], "label": preds[name]})
    assert len(submission) == len(sample_submission) and submission["id"].equals(sample_submission["id"])
    assert submission["label"].between(0, 1).all()
    submission.to_csv(SUBMISSION_DIR / filename, index=False)
    print("Wrote", SUBMISSION_DIR / filename)

Wrote ../submissions/04_qwen_all_train.csv
Wrote ../submissions/04_qwen_pseudo_label.csv


**Results:** leaderboard 0.979 for all train, 0.978 for pseudo-label
- Pseudo-labeling doesn't help: the 0.001 gap is within the noise of the 900 public rows
- One model on all 10,000 passages beats 03's 5-fold average (0.973), likely because each 03 fold model was missing whole topics (one never saw fiction, 36% of test)

## 6. Bigger Model

We can run the same setup with [`Qwen/Qwen2.5-1.5B`](https://huggingface.co/Qwen/Qwen2.5-1.5B) (Apache-2.0), which is about 3x larger

We had to get slightly creative to fit it, but this still runs on a Kaggle T4

In [13]:
BIG_MODEL = "Qwen/Qwen2.5-1.5B"
big = cached_run("qwen2.5-1.5b_all-train",
                 lambda: fit_predict(train_ids, y, test_ids, model_name=BIG_MODEL, checkpointing=True))

preds["1.5B all train"] = big
pd.DataFrame({
    "corr. with 0.5B all train": {name: spearman(p, control) for name, p in preds.items()},
    "test % predicted AI": {name: (p > 0.5).mean() * 100 for name, p in preds.items()},
})

qwen2.5-1.5b_all-train: loading saved predictions from ../results/04


,corr. with 0.5B all train,test % predicted AI
03 (5-fold average),0.944,47.267
all train,1.000,49.567
pseudo-label,0.962,47.300
1.5B all train,0.929,49.033


We write the 1.5B alone and its rank average with the 0.5B all-train model

In [14]:
def rank_average(*preds):
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

for filename, labels in [("04_qwen15b_all_train.csv", big), ("04_qwen05b_plus_15b_rank_avg.csv", rank_average(control, big))]:
    submission = pd.DataFrame({"id": test["id"], "label": labels})
    assert len(submission) == len(sample_submission) and submission["id"].equals(sample_submission["id"])
    assert submission["label"].between(0, 1).all()
    submission.to_csv(SUBMISSION_DIR / filename, index=False)
    print("Wrote", SUBMISSION_DIR / filename)

Wrote ../submissions/04_qwen15b_all_train.csv
Wrote ../submissions/04_qwen05b_plus_15b_rank_avg.csv


## 7. Fixed validation split

1.5B also needs a score on the team's fixed validation split (`splits/val_ids.csv`): one more model trained on the 8,000 training passages and scored on all 10,000, same as 03 did for the 0.5B

It was trained on a Kaggle T4 (44 min) in a separate run, so the other models didn't retrain. Predictions are saved in `results/04/04_qwen2.5-1.5b_fixed_split.csv`

In [15]:
# the team's fixed 8,000 / 2,000 split
_, val_rows = train_test_split(train, test_size=2000, stratify=y, random_state=42)
is_val = train["id"].isin(val_rows["id"]).values
assert train.loc[is_val, "id"].sum() == 10_098_025, "split doesn't match splits/val_ids.csv"

split_path = RESULTS_DIR / "04_qwen2.5-1.5b_fixed_split.csv"
if split_path.exists():
    big_split = train[["id"]].merge(pd.read_csv(split_path), on="id")["qwen"].values
else:
    start = time.time()
    big_split = fit_predict([train_ids[i] for i in np.flatnonzero(~is_val)], y[~is_val], train_ids,
                            model_name=BIG_MODEL, checkpointing=True)
    pd.DataFrame({"id": train["id"], "is_val": is_val, "qwen": big_split}).to_csv(split_path, index=False)
    print(f"Trained in {(time.time() - start) / 60:.0f} min")

In [16]:
small_split = train[["id"]].merge(pd.read_csv("../results/03/03_qwen2.5-0.5b_fixed_split.csv"), on="id")["qwen"].values

pd.DataFrame([{
    "train AUROC": roc_auc_score(y[~is_val], p[~is_val]),
    "val AUROC": roc_auc_score(y[is_val], p[is_val]),
    "val errors @0.5": int(((p[is_val] > 0.5) != y[is_val]).sum()),
} for p in (small_split, big_split)], index=["0.5B (from 03)", "1.5B"]).style.format(precision=4)

,train AUROC,val AUROC,val errors @0.5
0.5B (from 03),0.9998,0.9986,16
1.5B,1.0000,0.9998,11
